# Setup

In [214]:
import numpy as np
import pandas as pd
import kagglehub
import os
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, Trainer, TrainingArguments
import torch
from llm_api import OpenRouterApi
from dotenv import load_dotenv, dotenv_values 
import os
import json
from tqdm import tqdm
from typing import Tuple
import random

load_dotenv()

True

# Params

In [215]:
DATASET_DIR = './data'

OUTPUT_FILE = "hy3_results.jsonl"

# LLM_API = OpenRouterApi(key = os.getenv("OPEN_ROUTER_KEY"), model="z-ai/glm-4.5-air:free")
# LLM_API = OpenRouterApi(key = os.getenv("OPEN_ROUTER_KEY"), model="nousresearch/hermes-3-llama-3.1-405b:free")
LLM_API = OpenRouterApi(key = os.getenv("OPEN_ROUTER_KEY"), model="tencent/hy3-preview:free")
TEMPERATURE: float = 0.2
MAX_TOKENS: int|None = 1024

EXAMPLES_AMT = 2
PROMPT = """
   You are a Named Entity Recognition system.

    Task: Tag each token using BIO format:
    - B-PER, I-PER
    - B-ORG, I-ORG
    - B-LOC, I-LOC
    - B-MISC, I-MISC
    - O for non-entities

    Rules:
    - Output ONLY valid JSON
    - Do not add explanations
    - If unsure, use "O".
    - Use english.
    
    Before returning:
    - Verify JSON is valid
    - Verify that sentences lengths match

    Example:
    [EXAMPLE]

    Now process the following input.

    Input:
    [INPUT]
"""

MAX_FAILS_AMT = 5

# Test API

In [216]:
LLM_API.send_message("Do you work correctly? Talk in english")

"Yes, I work correctly! I'm here to help you with any questions or tasks you have. Feel free to ask me anything. 😊"

# Dataset handling

In [217]:
if not os.path.exists(DATASET_DIR):
    os.makedirs(DATASET_DIR)
    
path = kagglehub.dataset_download("alaakhaled/conll003-englishversion", output_dir=DATASET_DIR)
print(path)
print(os.listdir(path))

./data
['.complete', 'metadata', 'test.txt', 'train.txt', 'valid.txt']


In [218]:
def load_conll_ner(file_path):
    sentences = []
    tokens, labels = [], []
    
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            
            if not line:
                if tokens:
                    sentences.append({"tokens": tokens, "ner_tags": labels})
                    tokens, labels = [], []
                continue
            
            if line.startswith("-DOCSTART-"):
                continue
            
            word, pos, chunk, ner = line.split()
            tokens.append(word)
            labels.append(ner)
    if tokens:
        sentences.append({"tokens": tokens, "ner_tags": labels})

    return sentences


train_data = load_conll_ner(f"{path}/train.txt")
val_data = load_conll_ner(f"{path}/valid.txt")
test_data  = load_conll_ner(f"{path}/test.txt")

train_dataset = Dataset.from_list(train_data)
val_dataset   = Dataset.from_list(val_data)
test_dataset  = Dataset.from_list(test_data)

print(f"Number of training sentences: {len(train_dataset)}")
print(f"Number of validation sentences: {len(val_dataset)}")
print(f"Number of test sentences: {len(test_dataset)}")

print("Example sentence:")
print(train_dataset[0])

Number of training sentences: 14041
Number of validation sentences: 3250
Number of test sentences: 3453
Example sentence:
{'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'ner_tags': ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']}


# Training/testing loop

In [219]:
def create_example_json(min_length_of_example:int = 3) -> str:
    filtered = [ex for ex in train_dataset if len(ex["tokens"]) >= min_length_of_example]
    sorted_dataset = sorted(filtered, key=lambda ex: len(ex["tokens"]))
    shortest_pool = sorted_dataset[:EXAMPLES_AMT * 3]
    examples = random.sample(shortest_pool, EXAMPLES_AMT)

    example_str = ''
    for idx, ex in enumerate(examples):
        input_ = ex["tokens"]
        output_ = ex["ner_tags"]
        example_str += f'''
        {idx+1})
        Input:
        {json.dumps(input_, indent=None)}
        Output:
        {json.dumps(output_, indent=None)}
        '''

    return example_str

example_json = create_example_json()
print("Example JSON:")
print(example_json)

Example JSON:

        1)
        Input:
        ["Attendance", ":", "8,000"]
        Output:
        ["O", "O", "O"]
        
        2)
        Input:
        ["68", "Steve", "Stricker"]
        Output:
        ["O", "B-PER", "I-PER"]
        


In [220]:
def prepare_prompt_and_sample(sample_idx, is_print=False) -> str:
    sample = test_dataset[sample_idx]
    prompt = PROMPT.replace("[EXAMPLE]", example_json).replace("[INPUT]", json.dumps(sample["tokens"], indent=None))
    
    if sample_idx == 0 and is_print:
        print("=======================================================")
        print(prompt)
        print("=======================================================\n")
    
    return prompt, sample


def parse_response(sample_input, chat_response) -> Tuple[list, str]:
    try:
        pred = json.loads(chat_response)
    except json.JSONDecodeError:
        return None, "Can not parse response to json"
    
    if len(pred) != len(sample_input["tokens"]):
        return None, f"Predicted tags for sentence do not match tokens length"
        
    return pred, ""


print("Test prompt preparation:")
prompt, sample_input = prepare_prompt_and_sample(0, is_print=True)

Test prompt preparation:

   You are a Named Entity Recognition system.

    Task: Tag each token using BIO format:
    - B-PER, I-PER
    - B-ORG, I-ORG
    - B-LOC, I-LOC
    - B-MISC, I-MISC
    - O for non-entities

    Rules:
    - Output ONLY valid JSON
    - Do not add explanations
    - If unsure, use "O".
    - Use english.

    Before returning:
    - Verify JSON is valid
    - Verify that sentences lengths match

    Example:
    
        1)
        Input:
        ["Attendance", ":", "8,000"]
        Output:
        ["O", "O", "O"]
        
        2)
        Input:
        ["68", "Steve", "Stricker"]
        Output:
        ["O", "B-PER", "I-PER"]
        

    Now process the following input.

    Input:
    ["SOCCER", "-", "JAPAN", "GET", "LUCKY", "WIN", ",", "CHINA", "IN", "SURPRISE", "DEFEAT", "."]




In [221]:
def try_load_results_from_file(file_path):
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            return [json.loads(line) for line in f]
    return []

In [222]:
results = try_load_results_from_file(OUTPUT_FILE)
pbar = tqdm(range(len(results), len(test_dataset)))

for beg_idx in pbar:
    pbar.set_description(f"Processing sentence {beg_idx}")
    
    prompt, sample_input = prepare_prompt_and_sample(beg_idx)

    fails_amt = 0
    while True:
        response = LLM_API.send_message(prompt, 
                                        is_reasoning=fails_amt >= 2, 
                                        temperature=TEMPERATURE, 
                                        max_tokens=MAX_TOKENS * (fails_amt+1) if fails_amt < 2 else None,
                                        attempts_amt = 15)
        response_parsed, message = parse_response(sample_input, response)
        
        if response_parsed is None:
            fails_amt += 1
            
            if fails_amt >= MAX_FAILS_AMT:
                item = {"tokens": ["N/A"], 
                        "true_tags": ["N/A"],
                        "pred_tags": ["N/A"]}
                results.append(item)
                break
            
            pbar.set_description(f"Processing sentence {beg_idx}, fails: {fails_amt}")
            continue
        
        item = {"tokens": sample_input["tokens"], 
                "true_tags": sample_input["ner_tags"],
                "pred_tags": response_parsed}
        results.append(item)
        
        break
    
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for item in results:
            f.write(json.dumps(item) + "\n")   

Processing sentence 147:   1%|          | 29/3335 [01:48<3:26:33,  3.75s/it]          


KeyboardInterrupt: 

In [223]:
llm_results = try_load_results_from_file(OUTPUT_FILE)
llm_results = [item for item in llm_results if item["true_tags"] != ["N/A"]]

print(f"Total sentences processed: {len(llm_results)}")

true_tags = [item["true_tags"] for item in llm_results]
pred_tags = [item["pred_tags"] for item in llm_results]

print("LLM NER Classification Report:")
print(classification_report(true_tags, pred_tags))

Total sentences processed: 147
LLM NER Classification Report:
              precision    recall  f1-score   support

         LOC       0.87      0.99      0.93        88
        MISC       0.48      0.50      0.49        26
         ORG       0.82      0.69      0.75        13
         PER       0.99      0.97      0.98       174

   micro avg       0.90      0.92      0.91       301
   macro avg       0.79      0.79      0.79       301
weighted avg       0.91      0.92      0.91       301

